# PySpark practice 14: Structured Streaming

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/pyspark/pyspark-practice-14-structured-streaming.ipynb)

9 exercises on file streams, incremental runs, windows and watermarks, deduplication, stream joins, foreachBatch, output modes and progress metrics. Each exercise names the [reference entries](pyspark-reference-14-structured-streaming.ipynb) it practises, shows its inputs and the exact expected output, and hides a reference solution you can reveal when you are done or stuck.

*Last reviewed: November 2026, against Apache Spark 4.0.*

In [ ]:
# Setup: installs the packages this notebook needs when they are missing (e.g. in Colab).
!pip install -q pyspark~=4.0.0

## Setup and sample data

A local session, the shared `customers`, `orders`, `order_items`, `products` and `events` tables (customer 4 has no region, customers 7 and 8 never ordered, `SKU-4014` has no product row, one event is an exact retry), and the helpers from [PySpark reference 14](pyspark-reference-14-structured-streaming.ipynb#setup-and-sample-data). Exercises that need other data create it in their input cell. Streaming helpers: `land`, `run_available_now`, `new_dir`, `ev` and `EVENT_SCHEMA`.

In [ ]:
import os
import sys
import tempfile
import time

from pyspark.sql import SparkSession

os.environ["PYSPARK_PYTHON"] = sys.executable  # Python workers use this notebook's interpreter
os.environ["TZ"] = "UTC"  # the sample timestamps are UTC
time.tzset()
WORK_DIR = tempfile.mkdtemp(prefix="pyspark-")  # scratch space for the files the lessons write

spark = (
    SparkSession.builder.master("local[2]")
    .config("spark.sql.shuffle.partitions", "4")  # the default of 200 is far too many for tiny data
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.warehouse.dir", os.path.join(WORK_DIR, "warehouse"))
    .config("spark.ui.enabled", "false")
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version)

In [ ]:
from datetime import date, datetime
from decimal import Decimal as D

from pyspark.sql import Window
from pyspark.sql import functions as F

customers = spark.createDataFrame([
    (1, "Acme Corp", "North", date(2024, 1, 3)),
    (2, "Bright Retail", "North", date(2024, 2, 11)),
    (3, "Cyan Traders", "South", date(2024, 3, 15)),
    (4, "Delta Grocers", None, date(2024, 5, 2)),  # region never arrived from the CRM
    (5, "Echo Supplies", "West", date(2024, 6, 20)),
    (6, "Foxtrot Foods", "West", date(2024, 8, 1)),
    (7, "Gamma Sports", "East", date(2024, 9, 14)),  # 7 and 8 have never ordered
    (8, "Hotel Homeware", "East", date(2024, 11, 5)),
], "customer_id int, name string, region string, signup_date date")

orders = spark.createDataFrame([
    (1, 1, "paid", datetime(2026, 1, 5, 9, 15), D("120.00")),
    (2, 1, "shipped", datetime(2026, 1, 5, 10, 5), D("84.50")),
    (3, 1, "returned", datetime(2026, 1, 5, 11, 40), D("30.00")),
    (4, 2, "paid", datetime(2026, 1, 6, 15, 20), D("260.00")),
    (5, 2, "placed", datetime(2026, 1, 7, 8, 5), D("55.25")),
    (6, 3, "shipped", datetime(2026, 1, 7, 17, 45), D("149.00")),
    (7, 3, "paid", datetime(2026, 1, 9, 12, 10), D("310.00")),
    (8, 4, "returned", datetime(2026, 1, 10, 19, 30), D("89.90")),
    (9, 4, "shipped", datetime(2026, 1, 12, 8, 55), D("1180.00")),
    (10, 5, "paid", datetime(2026, 1, 12, 13, 0), D("74.00")),
    (11, 6, "placed", datetime(2026, 1, 13, 10, 25), D("210.00")),
    (12, 6, "shipped", datetime(2026, 1, 14, 0, 30), D("96.40")),  # just after midnight UTC
], "order_id int, customer_id int, status string, order_ts timestamp, amount decimal(12,2)")

order_items = spark.createDataFrame([
    (1, "SKU-4001", 2, D("12.50")), (1, "SKU-4007", 1, D("95.00")),
    (2, "SKU-4002", 1, D("34.50")), (2, "SKU-4009", 2, D("25.00")),
    (3, "SKU-4002", 1, D("30.00")),
    (4, "SKU-4003", 1, D("149.00")), (4, "SKU-4004", 2, D("55.50")),
    (5, "SKU-4005", 1, D("55.25")),
    (6, "SKU-4003", 1, D("149.00")),
    (7, "SKU-4006", 1, D("189.00")), (7, "SKU-4008", 1, D("121.00")),
    (8, "SKU-4010", 1, D("89.90")),
    (9, "SKU-4011", 20, D("59.00")),
    (10, "SKU-4012", 2, D("37.00")),
    (11, "SKU-4013", 1, D("210.00")),
    (12, "SKU-4014", 1, D("41.40")), (12, "SKU-4005", 1, D("55.00")),
], "order_id int, sku string, qty int, unit_price decimal(8,2)")

products = spark.createDataFrame([
    ("SKU-4001", "Kitchen", D("12.50")), ("SKU-4002", "Kitchen", D("34.50")),
    ("SKU-4003", "Garden", D("149.00")), ("SKU-4004", "Garden", D("55.50")),
    ("SKU-4005", "Pantry", D("55.25")), ("SKU-4006", "Outdoor", D("189.00")),
    ("SKU-4007", "Kitchen", D("95.00")), ("SKU-4008", "Outdoor", D("121.00")),
    ("SKU-4009", "Pantry", D("25.00")), ("SKU-4010", "Garden", D("89.90")),
    ("SKU-4011", "Pantry", D("59.00")), ("SKU-4012", "Kitchen", D("37.00")),
    ("SKU-4013", "Outdoor", D("210.00")),  # SKU-4014 is missing
], "sku string, category string, list_price decimal(8,2)")

events = spark.createDataFrame([
    (1, "view", datetime(2026, 1, 6, 9, 0)), (1, "add_to_cart", datetime(2026, 1, 6, 9, 2)),
    (1, "purchase", datetime(2026, 1, 6, 9, 10)), (2, "view", datetime(2026, 1, 7, 10, 0)),
    (2, "purchase", datetime(2026, 1, 7, 10, 40)), (3, "view", datetime(2026, 1, 6, 11, 15)),
    (3, "view", datetime(2026, 1, 8, 11, 20)), (4, "add_to_cart", datetime(2026, 1, 9, 20, 5)),
    (4, "purchase", datetime(2026, 1, 10, 8, 30)), (5, "view", datetime(2026, 1, 11, 14, 45)),
    (5, "view", datetime(2026, 1, 12, 9, 0)), (5, "purchase", datetime(2026, 1, 12, 12, 30)),
    (5, "view", datetime(2026, 1, 12, 9, 0)),  # an SDK retry committed this event twice
], "user_id int, event string, ts timestamp")

import contextlib
import io
import logging
import re


def plan_lines(df, mode="simple"):
    """The lines df.explain(mode) prints."""
    buffer = io.StringIO()
    with contextlib.redirect_stdout(buffer):
        df.explain(mode)
    return buffer.getvalue().splitlines()


def final_plan(df):
    """Run df, then return the plan Spark actually used (AQE's final plan when there is one)."""
    df.collect()
    text = "\n".join(plan_lines(df))
    if "== Final Plan ==" in text:
        text = text.split("== Final Plan ==")[1].split("== Initial Plan ==")[0]
    return text


@contextlib.contextmanager
def expected_error():
    """Run a block that is meant to fail and print the error class instead of a stack trace."""
    spark.sparkContext.setLogLevel("OFF")
    for logger in ("DataFrameQueryContextLogger", "SQLQueryContextLogger"):
        logging.getLogger(logger).setLevel(logging.CRITICAL)
    try:
        yield
    except Exception as error:
        lines = [line for line in str(error).splitlines() if line.strip()]
        message = next((line for line in lines if re.search(r"\[[A-Z][A-Z0-9_.]+\]", line)), lines[0] if lines else "")
        print(type(error).__name__ + ":", message.strip().split(". ")[0][:110])
    finally:
        spark.sparkContext.setLogLevel("ERROR")

import json
import os
import uuid

EVENT_SCHEMA = "event_id int, user_id int, event string, amount double, ts timestamp"


def new_dir(name):
    """A fresh directory under WORK_DIR, so every example starts clean."""
    path = os.path.join(WORK_DIR, f"{name}-{uuid.uuid4().hex[:6]}")
    os.makedirs(path)
    return path


def land(directory, rows, name=None):
    """Write one JSON Lines file into a landing directory, like a producer dropping a batch."""
    name = name or f"batch-{uuid.uuid4().hex[:8]}.json"
    with open(os.path.join(directory, name), "w") as f:
        for r in rows:
            f.write(json.dumps(r) + "\n")


def run_available_now(writer):
    """Start a streaming write that processes everything available, then stops; return the query."""
    query = writer.trigger(availableNow=True).start()
    query.awaitTermination()
    return query


def ev(event_id, user_id, event, amount, ts):
    return {"event_id": event_id, "user_id": user_id, "event": event, "amount": amount, "ts": ts}

## Exercises

4 easy, 3 medium, 2 hard. Write your answer in the empty cell under each exercise as a DataFrame called `result`, run it, and compare it with the expected output (columns, values and row order).

| # | exercise | difficulty | practises |
| --- | --- | --- | --- |
| 1 | [Count events from a file stream](#exercise-1-count-events-from-a-file-stream) | Easy | [`spark.readStream`: `DataStreamReader`](pyspark-reference-14-structured-streaming.ipynb#sparkreadstream-datastreamreader), [Triggers](pyspark-reference-14-structured-streaming.ipynb#triggers) |
| 2 | [Incremental runs pick up only new files](#exercise-2-incremental-runs-pick-up-only-new-files) | Medium | [Checkpoints and exactly-once](pyspark-reference-14-structured-streaming.ipynb#checkpoints-and-exactly-once) |
| 3 | [Ten-minute windowed counts](#exercise-3-ten-minute-windowed-counts) | Easy | [Event-time windows](pyspark-reference-14-structured-streaming.ipynb#event-time-windows) |
| 4 | [Late data and the watermark](#exercise-4-late-data-and-the-watermark) | Hard | [`withWatermark(eventTime, delayThreshold)`](pyspark-reference-14-structured-streaming.ipynb#withwatermarkeventtime-delaythreshold) |
| 5 | [Drop retried events in a stream](#exercise-5-drop-retried-events-in-a-stream) | Medium | [Deduplication: `dropDuplicates` and `dropDuplicatesWithinWatermark`](pyspark-reference-14-structured-streaming.ipynb#deduplication-dropduplicates-and-dropduplicateswithinwatermark) |
| 6 | [Enrich a stream with a static dimension](#exercise-6-enrich-a-stream-with-a-static-dimension) | Easy | [Stream-static and stream-stream joins](pyspark-reference-14-structured-streaming.ipynb#stream-static-and-stream-stream-joins) |
| 7 | [Upsert latest state per user with foreachBatch](#exercise-7-upsert-latest-state-per-user-with-foreachbatch) | Hard | [`writeStream.foreachBatch(func)`](pyspark-reference-14-structured-streaming.ipynb#writestreamforeachbatchfunc) |
| 8 | [Update mode emits only changed rows](#exercise-8-update-mode-emits-only-changed-rows) | Medium | [Output modes](pyspark-reference-14-structured-streaming.ipynb#output-modes) |
| 9 | [Read progress metrics](#exercise-9-read-progress-metrics) | Easy | [`StreamingQuery` status and progress](pyspark-reference-14-structured-streaming.ipynb#streamingquery-status-and-progress) |

## Exercise 1: Count events from a file stream

**Difficulty:** Easy · about 5 min · **Practises:** [`spark.readStream`: `DataStreamReader`](pyspark-reference-14-structured-streaming.ipynb#sparkreadstream-datastreamreader), [Triggers](pyspark-reference-14-structured-streaming.ipynb#triggers)

*The simplest incremental job: read new files, aggregate, stop.*

Stream the JSON files in `lnd` with `EVENT_SCHEMA`, count events per `event` in complete mode into memory table `p14_counts` with `availableNow`, and return the table ordered by `event`.

In [ ]:
lnd = new_dir("p14_lnd")
land(lnd, [ev(1, 1, "view", None, "2026-01-06 09:00:00"), ev(2, 1, "buy", 5.0, "2026-01-06 09:05:00"), ev(3, 2, "view", None, "2026-01-06 09:06:00")])

**Expected output**

| event | count |
| --- | --- |
| buy | 1 |
| view | 2 |

In [ ]:
# Write your solution here: build a DataFrame called result.

In [ ]:
counts = spark.readStream.schema(EVENT_SCHEMA).json(lnd).groupBy("event").count()
run_available_now(counts.writeStream.format("memory").queryName("p14_counts").outputMode("complete").option("checkpointLocation", new_dir("chk")))
result = spark.table("p14_counts").orderBy("event")
result.show()

## Exercise 2: Incremental runs pick up only new files

**Difficulty:** Medium · about 10 min · **Practises:** [Checkpoints and exactly-once](pyspark-reference-14-structured-streaming.ipynb#checkpoints-and-exactly-once)

*Scheduled availableNow jobs must not reprocess old files.*

Run an `availableNow` stream from `lnd2` to Parquet at `out2` twice with the same checkpoint, landing one more file between runs. Return `run` and `input_rows` (rows the run read).

In [ ]:
lnd2, out2, chk2 = new_dir("p14_lnd2"), new_dir("p14_out2"), new_dir("p14_chk2")
land(lnd2, [ev(1, 1, "view", None, "2026-01-06 09:00:00"), ev(2, 2, "view", None, "2026-01-06 09:01:00")])

**Expected output**

| run | input_rows |
| --- | --- |
| 1 | 2 |
| 2 | 1 |

In [ ]:
# Write your solution here: build a DataFrame called result.

In [ ]:
def run_once():
    q = run_available_now(spark.readStream.schema(EVENT_SCHEMA).json(lnd2).writeStream.format("parquet").option("path", out2).option("checkpointLocation", chk2))
    return sum(p["numInputRows"] for p in q.recentProgress)

first = run_once()
land(lnd2, [ev(3, 3, "buy", 7.0, "2026-01-06 09:30:00")])
second = run_once()
result = spark.createDataFrame([(1, first), (2, second)], "run int, input_rows long")
result.show()

## Exercise 3: Ten-minute windowed counts

**Difficulty:** Easy · about 5 min · **Practises:** [Event-time windows](pyspark-reference-14-structured-streaming.ipynb#event-time-windows)

*Per-interval metrics from event time, not arrival time.*

Count events per 10-minute tumbling window of `ts` from `lnd3` (complete mode, memory sink). Return `start` and `events`, ordered by `start`.

In [ ]:
lnd3 = new_dir("p14_lnd3")
land(lnd3, [ev(1, 1, "view", None, "2026-01-06 09:01:00"), ev(2, 2, "view", None, "2026-01-06 09:09:00"), ev(3, 1, "buy", 2.0, "2026-01-06 09:15:00")])

**Expected output**

| start | events |
| --- | --- |
| 2026-01-06 09:00:00 | 2 |
| 2026-01-06 09:10:00 | 1 |

In [ ]:
# Write your solution here: build a DataFrame called result.

In [ ]:
w = spark.readStream.schema(EVENT_SCHEMA).json(lnd3).groupBy(F.window("ts", "10 minutes")).agg(F.count("*").alias("events"))
run_available_now(w.writeStream.format("memory").queryName("p14_win").outputMode("complete").option("checkpointLocation", new_dir("chk")))
result = spark.table("p14_win").select(F.col("window.start").alias("start"), "events").orderBy("start")
result.show()

## Exercise 4: Late data and the watermark

**Difficulty:** Hard · about 20 min · **Practises:** [`withWatermark(eventTime, delayThreshold)`](pyspark-reference-14-structured-streaming.ipynb#withwatermarkeventtime-delaythreshold)

*Watermarks bound state and decide which late events still count.*

Aggregate 10-minute window counts from `lnd4` with a 5-minute watermark in append mode to Parquet. Run once with the first file, land the second file (containing one event late beyond the watermark), and run again. Return `start` and `events` from the output, ordered by `start`.

In [ ]:
lnd4, out4, chk4 = new_dir("p14_lnd4"), new_dir("p14_out4"), new_dir("p14_chk4")
first_file = [ev(1, 1, "view", None, "2026-01-06 09:01:00"), ev(2, 1, "view", None, "2026-01-06 09:25:00")]
second_file = [ev(3, 2, "view", None, "2026-01-06 09:04:00"), ev(4, 2, "view", None, "2026-01-06 09:45:00")]

**Expected output**

| start | events |
| --- | --- |
| 2026-01-06 09:00:00 | 1 |
| 2026-01-06 09:20:00 | 1 |

In [ ]:
# Write your solution here: build a DataFrame called result.

In [ ]:
agg = (spark.readStream.schema(EVENT_SCHEMA).json(lnd4).withWatermark("ts", "5 minutes")
       .groupBy(F.window("ts", "10 minutes")).agg(F.count("*").alias("events")))
run = lambda: run_available_now(agg.writeStream.format("parquet").option("path", out4).option("checkpointLocation", chk4).outputMode("append"))
land(lnd4, first_file)
run()
land(lnd4, second_file)
run()
result = spark.read.parquet(out4).select(F.col("window.start").alias("start"), "events").orderBy("start")
result.show()

## Exercise 5: Drop retried events in a stream

**Difficulty:** Medium · about 10 min · **Practises:** [Deduplication: `dropDuplicates` and `dropDuplicatesWithinWatermark`](pyspark-reference-14-structured-streaming.ipynb#deduplication-dropduplicates-and-dropduplicateswithinwatermark)

*Producers retry; the stream must count each event once with bounded state.*

Deduplicate `lnd5` by `event_id` within a 1-hour watermark and write to memory table `p14_dedup`. Return `event_id` and `event`, ordered by `event_id`.

In [ ]:
lnd5 = new_dir("p14_lnd5")
land(lnd5, [ev(1, 1, "view", None, "2026-01-06 09:00:00"), ev(1, 1, "view", None, "2026-01-06 09:00:00"), ev(2, 2, "buy", 9.0, "2026-01-06 09:02:00"), ev(2, 2, "buy", 9.0, "2026-01-06 09:02:00")])

**Expected output**

| event_id | event |
| --- | --- |
| 1 | view |
| 2 | buy |

In [ ]:
# Write your solution here: build a DataFrame called result.

In [ ]:
d = spark.readStream.schema(EVENT_SCHEMA).json(lnd5).withWatermark("ts", "1 hour").dropDuplicatesWithinWatermark(["event_id"])
run_available_now(d.writeStream.format("memory").queryName("p14_dedup").outputMode("append").option("checkpointLocation", new_dir("chk")))
result = spark.table("p14_dedup").select("event_id", "event").orderBy("event_id")
result.show()

## Exercise 6: Enrich a stream with a static dimension

**Difficulty:** Easy · about 5 min · **Practises:** [Stream-static and stream-stream joins](pyspark-reference-14-structured-streaming.ipynb#stream-static-and-stream-stream-joins)

*Most streaming pipelines attach dimension attributes to events.*

Join events from `lnd6` to `customers` on `user_id = customer_id` (left join, stream on the left) and write to memory table `p14_enriched`. Return `event_id`, `name`, ordered by `event_id`.

In [ ]:
lnd6 = new_dir("p14_lnd6")
land(lnd6, [ev(1, 1, "view", None, "2026-01-06 09:00:00"), ev(2, 99, "view", None, "2026-01-06 09:01:00")])

**Expected output**

| event_id | name |
| --- | --- |
| 1 | Acme Corp |
| 2 | null |

In [ ]:
# Write your solution here: build a DataFrame called result.

In [ ]:
s = spark.readStream.schema(EVENT_SCHEMA).json(lnd6)
joined = s.join(customers, s.user_id == customers.customer_id, "left").select("event_id", "name")
run_available_now(joined.writeStream.format("memory").queryName("p14_enriched").outputMode("append").option("checkpointLocation", new_dir("chk")))
result = spark.table("p14_enriched").orderBy("event_id")
result.show()

## Exercise 7: Upsert latest state per user with foreachBatch

**Difficulty:** Hard · about 20 min · **Practises:** [`writeStream.foreachBatch(func)`](pyspark-reference-14-structured-streaming.ipynb#writestreamforeachbatchfunc)

*foreachBatch is how streams feed upsert targets.*

Maintain the latest event per user at `target` with `foreachBatch`: each batch keeps the newest event per user and replaces those users' rows. Run twice (second file has a newer event for user 1). Return `user_id`, `event_id`, ordered by `user_id`.

In [ ]:
lnd7, chk7, target = new_dir("p14_lnd7"), new_dir("p14_chk7"), os.path.join(WORK_DIR, "p14_target")
land(lnd7, [ev(1, 1, "view", None, "2026-01-06 09:00:00"), ev(2, 2, "view", None, "2026-01-06 09:01:00")])

**Expected output**

| user_id | event_id |
| --- | --- |
| 1 | 3 |
| 2 | 2 |

In [ ]:
# Write your solution here: build a DataFrame called result.

In [ ]:
def upsert(batch_df, batch_id):
    latest = batch_df.withColumn("rn", F.row_number().over(Window.partitionBy("user_id").orderBy(F.desc("ts")))).where("rn = 1").select("user_id", "event_id", "ts")
    if os.path.exists(target):
        merged = spark.read.parquet(target).join(latest, "user_id", "left_anti").unionByName(latest)
    else:
        merged = latest
    rows = merged.collect()
    spark.createDataFrame(rows, merged.schema).write.mode("overwrite").parquet(target)

run = lambda: run_available_now(spark.readStream.schema(EVENT_SCHEMA).json(lnd7).writeStream.foreachBatch(upsert).option("checkpointLocation", chk7))
run()
land(lnd7, [ev(3, 1, "buy", 15.0, "2026-01-06 09:40:00")])
run()
result = spark.read.parquet(target).select("user_id", "event_id").orderBy("user_id")
result.show()

## Exercise 8: Update mode emits only changed rows

**Difficulty:** Medium · about 10 min · **Practises:** [Output modes](pyspark-reference-14-structured-streaming.ipynb#output-modes)

*Choosing the output mode decides how much is written per batch.*

Count events per user from `lnd8` in **update** mode with `foreachBatch`, collecting the rows each batch emits. Run twice (the second file only has user 2). Return `batch_run` (batch id + 1), `user_id` and `count` for every emitted row, ordered by `batch_run`, `user_id`.

In [ ]:
lnd8, chk8 = new_dir("p14_lnd8"), new_dir("p14_chk8")
land(lnd8, [ev(1, 1, "view", None, "2026-01-06 09:00:00"), ev(2, 2, "view", None, "2026-01-06 09:01:00")])
emitted = []

**Expected output**

| batch_run | user_id | count |
| --- | --- | --- |
| 1 | 1 | 1 |
| 1 | 2 | 1 |
| 2 | 2 | 2 |

In [ ]:
# Write your solution here: build a DataFrame called result.

In [ ]:
def capture(batch_df, batch_id):
    for r in batch_df.collect():
        emitted.append((batch_id + 1, r.user_id, r["count"]))

agg = spark.readStream.schema(EVENT_SCHEMA).json(lnd8).groupBy("user_id").count()
run = lambda: run_available_now(agg.writeStream.outputMode("update").foreachBatch(capture).option("checkpointLocation", chk8))
run()
land(lnd8, [ev(3, 2, "buy", 4.0, "2026-01-06 09:30:00")])
run()
result = spark.createDataFrame(emitted, "batch_run int, user_id int, count long").orderBy("batch_run", "user_id")
result.show()

## Exercise 9: Read progress metrics

**Difficulty:** Easy · about 5 min · **Practises:** [`StreamingQuery` status and progress](pyspark-reference-14-structured-streaming.ipynb#streamingquery-status-and-progress)

*Monitoring reads numInputRows and state size from query progress.*

Run a complete-mode count per `event` over `lnd9` with `availableNow`. Return one row with `input_rows` (sum of `numInputRows` over `recentProgress`) and `state_rows` (`numRowsTotal` of the state operator in the last progress).

In [ ]:
lnd9 = new_dir("p14_lnd9")
land(lnd9, [ev(i, i % 2, "view" if i % 3 else "buy", None, "2026-01-06 09:00:00") for i in range(1, 7)])

**Expected output**

| input_rows | state_rows |
| --- | --- |
| 6 | 2 |

In [ ]:
# Write your solution here: build a DataFrame called result.

In [ ]:
q = run_available_now(spark.readStream.schema(EVENT_SCHEMA).json(lnd9).groupBy("event").count()
                      .writeStream.format("memory").queryName("p14_prog").outputMode("complete").option("checkpointLocation", new_dir("chk")))
input_rows = sum(p["numInputRows"] for p in q.recentProgress)
state_rows = q.lastProgress["stateOperators"][0]["numRowsTotal"]
result = spark.createDataFrame([(input_rows, state_rows)], "input_rows long, state_rows long")
result.show()

**Next:** look anything up in [PySpark reference 14: Structured Streaming](pyspark-reference-14-structured-streaming.ipynb), or test yourself across topics with the [PySpark coding challenges](pyspark-coding-challenges.ipynb).